In [5]:
import os 
print(os.getcwd())

import glob

all_csv = glob.glob("*.csv")
print(all_csv)

C:\Users\Shawn\PROJECT\Stocksense\Data\Twitter\cleaned
['AAPL_cleaned.csv', 'AAPL_cleaned_sentiment.csv', 'AMZN_cleaned.csv', 'AMZN_cleaned_sentiment.csv', 'ARKK_cleaned.csv', 'ARKK_cleaned_sentiment.csv', 'ATER_cleaned.csv', 'ATER_cleaned_sentiment.csv', 'GOOGL_cleaned.csv', 'GOOGL_cleaned_sentiment.csv', 'MSFT_cleaned.csv', 'MSFT_cleaned_sentiment.csv', 'Russell 1000_cleaned.csv', 'Russell 1000_cleaned_sentiment.csv', 'TSLA_cleaned.csv', 'TSLA_cleaned_sentiment.csv', 'VXRT_cleaned.csv', 'VXRT_cleaned_sentiment.csv']


# Flair

In [ ]:
#!pip install flair

In [24]:
import pandas as pd
import flair
flair_sentiment = flair.models.TextClassifier.load('en-sentiment')  # Load model


def flair_senitment(ticketdf):
    for index, row in ticketdf.iterrows():
        s = flair.data.Sentence(row["cleaned Tweet"])  # Retrieve title of post
        flair_sentiment.predict(s) 
        sentiment = str(s.labels[0]).split()[0]
        score = str(s.labels[0]).split()[1][1:-1]

        ticketdf.at[index,'flair_sentiment'] = sentiment
        ticketdf.at[index,'flair_sentiment_score'] = score

    return ticketdf

2021-10-19 00:00:26,070 loading file C:\Users\Shawn\.flair\models\sentiment-en-mix-distillbert_4.pt


# Vader

In [25]:
# import nltk
# nltk.download('vader_lexicon')

In [26]:
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

def vader_senitment(ticketdf):
    for index, row in ticketdf.iterrows():

        vs = sia.polarity_scores(row["cleaned Tweet"])

        score = vs["compound"]
        del vs['compound']
        sentiment = max(vs, key=vs.get)

        ticketdf.at[index,'vader_sentiment'] = sentiment
        ticketdf.at[index,'vader_score'] = score
    

    return ticketdf

# Finbert

In [27]:
# !pip install happytransformer

In [28]:
import pandas as pd
from happytransformer import HappyTextClassification 
happy_tc = HappyTextClassification("BERT", "ProsusAI/finbert", num_labels=3)


def finbert_senitment(ticketdf):

    for index, row in ticketdf.iterrows():
        result = happy_tc.classify_text(row["cleaned Tweet"])
        sentiment = result.label
        score = result.score

        ticketdf.at[index,'finbert_sentiment'] = sentiment
        ticketdf.at[index,'finbert_score'] = score
    
    return ticketdf

10/19/2021 00:00:49 - INFO - happytransformer.happy_transformer -   Using model: cpu


In [29]:
for file in all_csv:
    file_df = pd.read_csv(file)
    file_df.head()
    print("======start======")
    file_df = flair_senitment(file_df) 
    print("flair done")
    print(file)
    file_df = vader_senitment(file_df)
    print("vader done")
    file_df = finbert_senitment(file_df)
    print("finbert done")
    filename = file.split(".")[0] + "_sentiment.csv"
    
    file_df.to_csv(filename,index=False)
    print(file,"done")

======start======


KeyboardInterrupt: 